In [1]:
import pandas as pd
import numpy as np

In [2]:
results = pd.read_csv("../data/processed/cleaned_results.csv")
results["date"] = pd.to_datetime(results["date"])
results = results.sort_values("date").reset_index(drop=True)

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,result_label
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,Draw,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,Home Win,1
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,Home Win,1
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,Draw,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,Home Win,1


In [3]:
home_matches = results[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "neutral",
    "result_label"
]].copy()

home_matches["team"] = home_matches["home_team"]
home_matches["opponent"] = home_matches["away_team"]
home_matches["goals_scored"] = home_matches["home_score"]
home_matches["goals_conceded"] = home_matches["away_score"]

home_matches["team_result"] = home_matches["result_label"]

In [4]:
away_matches = results[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "neutral",
    "result_label"
]].copy()

away_matches["team"] = away_matches["away_team"]
away_matches["opponent"] = away_matches["home_team"]
away_matches["goals_scored"] = away_matches["away_score"]
away_matches["goals_conceded"] = away_matches["home_score"]

away_matches["team_result"] = away_matches["result_label"].map({
    1: -1,
    0: 0,
    -1: 1
})

In [5]:
team_matches = pd.concat([home_matches, away_matches], ignore_index=True)

team_matches = team_matches[[
    "date",
    "team",
    "opponent",
    "goals_scored",
    "goals_conceded",
    "team_result",
    "tournament",
    "neutral"
]]

team_matches.head()

,date,team,opponent,goals_scored,goals_conceded,team_result,tournament,neutral
0,1872-11-30,Scotland,England,0.0,0.0,0,Friendly,False
1,1873-03-08,England,Scotland,4.0,2.0,1,Friendly,False
2,1874-03-07,Scotland,England,2.0,1.0,1,Friendly,False
3,1875-03-06,England,Scotland,2.0,2.0,0,Friendly,False
4,1876-03-04,Scotland,England,3.0,0.0,1,Friendly,False


In [6]:
team_matches["win"] = (team_matches["team_result"] == 1).astype(int)
team_matches["draw"] = (team_matches["team_result"] == 0).astype(int)
team_matches["loss"] = (team_matches["team_result"] == -1).astype(int)

team_matches.head()

,date,team,opponent,goals_scored,goals_conceded,team_result,tournament,neutral,win,draw,loss
0,1872-11-30,Scotland,England,0.0,0.0,0,Friendly,False,0,1,0
1,1873-03-08,England,Scotland,4.0,2.0,1,Friendly,False,1,0,0
2,1874-03-07,Scotland,England,2.0,1.0,1,Friendly,False,1,0,0
3,1875-03-06,England,Scotland,2.0,2.0,0,Friendly,False,0,1,0
4,1876-03-04,Scotland,England,3.0,0.0,1,Friendly,False,1,0,0


In [7]:
team_matches = team_matches.sort_values(["team", "date"]).reset_index(drop=True)

team_matches["recent_win_rate"] = (
    team_matches
    .groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_matches["recent_avg_goals_scored"] = (
    team_matches
    .groupby("team")["goals_scored"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_matches["recent_avg_goals_conceded"] = (
    team_matches
    .groupby("team")["goals_conceded"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [8]:
team_matches[team_matches["team"] == "Brazil"].tail(10)

,date,team,opponent,goals_scored,goals_conceded,team_result,tournament,neutral,win,draw,loss,recent_win_rate,recent_avg_goals_scored,recent_avg_goals_conceded
11160,2025-10-14,Brazil,Japan,2.0,3.0,-1,Kirin Cup,False,0,0,1,0.6,1.8,0.20
11161,2025-11-15,Brazil,Senegal,2.0,0.0,1,Friendly,True,1,0,0,0.6,2.2,0.80
11162,2025-11-18,Brazil,Tunisia,1.0,1.0,0,Friendly,True,0,1,0,0.6,2.4,0.80
11163,2026-03-26,Brazil,France,1.0,2.0,-1,Friendly,True,0,0,1,0.4,2.0,1.00
11164,2026-03-31,Brazil,Croatia,3.0,1.0,1,Friendly,True,1,0,0,0.4,2.2,1.20
11165,2026-05-31,Brazil,Panama,6.0,2.0,1,Friendly,False,1,0,0,0.4,1.8,1.40
11166,2026-06-06,Brazil,Egypt,2.0,1.0,1,Friendly,True,1,0,0,0.6,2.6,1.20
11167,2026-06-13,Brazil,Morocco,1.0,1.0,0,FIFA World Cup,True,0,1,0,0.6,2.6,1.40
11168,2026-06-19,Brazil,Haiti,NaN,NaN,0,FIFA World Cup,True,0,1,0,0.6,2.6,1.40
11169,2026-06-24,Brazil,Scotland,NaN,NaN,0,FIFA World Cup,True,0,1,0,0.6,3.0,1.25


In [9]:
home_features = team_matches[[
    "date",
    "team",
    "recent_win_rate",
    "recent_avg_goals_scored",
    "recent_avg_goals_conceded"
]].copy()

home_features = home_features.rename(columns={
    "team": "home_team",
    "recent_win_rate": "home_recent_win_rate",
    "recent_avg_goals_scored": "home_recent_avg_goals_scored",
    "recent_avg_goals_conceded": "home_recent_avg_goals_conceded"
})

In [10]:
away_features = team_matches[[
    "date",
    "team",
    "recent_win_rate",
    "recent_avg_goals_scored",
    "recent_avg_goals_conceded"
]].copy()

away_features = away_features.rename(columns={
    "team": "away_team",
    "recent_win_rate": "away_recent_win_rate",
    "recent_avg_goals_scored": "away_recent_avg_goals_scored",
    "recent_avg_goals_conceded": "away_recent_avg_goals_conceded"
})

In [11]:
model_data = results.merge(
    home_features,
    on=["date", "home_team"],
    how="left"
)

model_data = model_data.merge(
    away_features,
    on=["date", "away_team"],
    how="left"
)

model_data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,result_label,home_recent_win_rate,home_recent_avg_goals_scored,home_recent_avg_goals_conceded,away_recent_win_rate,away_recent_avg_goals_scored,away_recent_avg_goals_conceded
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,Draw,0,NaN,NaN,NaN,NaN,NaN,NaN
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,Home Win,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,Home Win,1,0.000000,1.000000,2.000000,0.500000,2.000000,1.000000
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,Draw,0,0.333333,1.666667,1.333333,0.333333,1.333333,1.666667
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,Home Win,1,0.250000,1.500000,1.750000,0.250000,1.750000,1.500000


In [12]:
model_data.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'result', 'result_label',
       'home_recent_win_rate', 'home_recent_avg_goals_scored',
       'home_recent_avg_goals_conceded', 'away_recent_win_rate',
       'away_recent_avg_goals_scored', 'away_recent_avg_goals_conceded'],
      dtype='object')

In [13]:
model_data["win_rate_difference"] = (
    model_data["home_recent_win_rate"] - model_data["away_recent_win_rate"]
)

model_data["attack_difference"] = (
    model_data["home_recent_avg_goals_scored"] - model_data["away_recent_avg_goals_scored"]
)

model_data["defense_difference"] = (
    model_data["home_recent_avg_goals_conceded"] - model_data["away_recent_avg_goals_conceded"]
)

In [14]:
model_data[[
    "home_recent_win_rate",
    "home_recent_avg_goals_scored",
    "home_recent_avg_goals_conceded",
    "away_recent_win_rate",
    "away_recent_avg_goals_scored",
    "away_recent_avg_goals_conceded",
    "win_rate_difference",
    "attack_difference",
    "defense_difference"
]].isnull().sum()

home_recent_win_rate              141
home_recent_avg_goals_scored      141
home_recent_avg_goals_conceded    141
away_recent_win_rate              195
away_recent_avg_goals_scored      195
away_recent_avg_goals_conceded    195
win_rate_difference               302
attack_difference                 302
defense_difference                302
dtype: int64

In [15]:
feature_columns = [
    "home_recent_win_rate",
    "home_recent_avg_goals_scored",
    "home_recent_avg_goals_conceded",
    "away_recent_win_rate",
    "away_recent_avg_goals_scored",
    "away_recent_avg_goals_conceded",
    "win_rate_difference",
    "attack_difference",
    "defense_difference"
]

model_data[feature_columns] = model_data[feature_columns].fillna(0)

In [16]:
model_data.to_csv("../data/processed/model_data.csv", index=False)